<a href="https://colab.research.google.com/github/Walnut235/olist-probabilistic-analysis/blob/main/notebooks/01_analisis_olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Probabilístico del E-Commerce Brasileño — Olist

## Machine Learning Probabilístico

**Dataset:** Brazilian E-Commerce Public Dataset by Olist

### Objetivo

Analizar el comportamiento del comercio electrónico de Olist mediante conceptos de probabilidad, estadística e información, con el fin de identificar patrones relevantes para la toma de decisiones.

### Conceptos a estudiar

1. Probabilidad condicional
2. Teorema de Bayes
3. Verosimilitud
4. Máxima verosimilitud (MLE)
5. Distribuciones paramétricas
6. Esperanza y varianza
7. Independencia y correlación
8. Prior y posterior
9. Entropía
10. Entropía cruzada
11. Divergencia KL

# Cargar datos

**Importar librerias**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

**Descarga del dataset de Olist**

In [4]:
!pip install -q kagglehub
# conectarse a kaggle
import kagglehub
import os

# Descargar el dataset de Olist
dataset_path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print("Dataset descargado en:")
print(dataset_path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset descargado en:
/kaggle/input/brazilian-ecommerce


**Carga de las tablas**

In [6]:
# Diccionario para almacenar las tablas
data = {}
# Guardar las tablas en el diccionario
for file in files:
    file_path = os.path.join(dataset_path, file)
    table_name = file.replace(".csv", "")

    data[table_name] = pd.read_csv(file_path)

print("Tablas cargadas correctamente:")
for name, df in data.items():
    print(f"{name}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Tablas cargadas correctamente:
olist_customers_dataset: 99,441 filas × 5 columnas
olist_sellers_dataset: 3,095 filas × 4 columnas
olist_order_reviews_dataset: 99,224 filas × 7 columnas
olist_order_items_dataset: 112,650 filas × 7 columnas
olist_products_dataset: 32,951 filas × 9 columnas
olist_geolocation_dataset: 1,000,163 filas × 5 columnas
product_category_name_translation: 71 filas × 2 columnas
olist_orders_dataset: 99,441 filas × 8 columnas
olist_order_payments_dataset: 103,886 filas × 5 columnas


**Conocer las tablas**

In [10]:
for name, df in data.items(): # recorrer el diccionario
    print(f"\n{'=' * 60}")
    print(name) # nombre de la tabla
    print(f"{'=' * 60}")
    print("Columnas:")

    for column in df.columns: # mostrar las columnas del df
        print(f"  - {column}")



olist_customers_dataset
Columnas:
  - customer_id
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state

olist_sellers_dataset
Columnas:
  - seller_id
  - seller_zip_code_prefix
  - seller_city
  - seller_state

olist_order_reviews_dataset
Columnas:
  - review_id
  - order_id
  - review_score
  - review_comment_title
  - review_comment_message
  - review_creation_date
  - review_answer_timestamp

olist_order_items_dataset
Columnas:
  - order_id
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value

olist_products_dataset
Columnas:
  - product_id
  - product_category_name
  - product_name_lenght
  - product_description_lenght
  - product_photos_qty
  - product_weight_g
  - product_length_cm
  - product_height_cm
  - product_width_cm

olist_geolocation_dataset
Columnas:
  - geolocation_zip_code_prefix
  - geolocation_lat
  - geolocation_lng
  - geolocation_city
  - geolocation_state

product_category_name_tr

**Verificar relaciones (1:N, 1:1, M:N)**

In [8]:
# ¿Cada customer_id aparece una sola vez?
print("CUSTOMERS")
print("customer_id únicos:", data["olist_customers_dataset"]["customer_id"].nunique())
print("filas:", len(data["olist_customers_dataset"]))

print("\nORDERS")
print("order_id únicos:", data["olist_orders_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_orders_dataset"]))

print("\nPRODUCTS")
print("product_id únicos:", data["olist_products_dataset"]["product_id"].nunique())
print("filas:", len(data["olist_products_dataset"]))

print("\nSELLERS")
print("seller_id únicos:", data["olist_sellers_dataset"]["seller_id"].nunique())
print("filas:", len(data["olist_sellers_dataset"]))

print("\nREVIEWS")
print("order_id únicos:", data["olist_order_reviews_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_order_reviews_dataset"]))

print("\nORDER ITEMS")
print("order_id únicos:", data["olist_order_items_dataset"]["order_id"].nunique())
print("filas:", len(data["olist_order_items_dataset"]))

CUSTOMERS
customer_id únicos: 99441
filas: 99441

ORDERS
order_id únicos: 99441
filas: 99441

PRODUCTS
product_id únicos: 32951
filas: 32951

SELLERS
seller_id únicos: 3095
filas: 3095

REVIEWS
order_id únicos: 98673
filas: 99224

ORDER ITEMS
order_id únicos: 98666
filas: 112650


**Union de las tablas**

In [22]:
# Guardar cada tabla en un df individual
orders = data["olist_orders_dataset"].copy()
customers = data["olist_customers_dataset"].copy()
items = data["olist_order_items_dataset"].copy()
products = data["olist_products_dataset"].copy()
sellers = data["olist_sellers_dataset"].copy()
reviews = data["olist_order_reviews_dataset"].copy()
payments = data["olist_order_payments_dataset"].copy()
geolocation = data["olist_geolocation_dataset"].copy()
category_translation = data["product_category_name_translation"].copy()

In [27]:
# ============================================
# 9. CONSTRUCCIÓN DEL DATAFRAME MAESTRO
# ============================================

df_master = orders.copy()

print("INICIAL")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

INICIAL
Filas: 99441
Pedidos únicos: 99441


In [28]:
df_master = df_master.merge(
    customers,
    on="customer_id",
    how="left",
    validate="one_to_one" #Relación 1:1
)

print("DESPUÉS DE CUSTOMERS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE CUSTOMERS
Filas: 99441
Pedidos únicos: 99441


In [29]:
df_master = df_master.merge(
    items,
    on="order_id",
    how="left",
    validate="one_to_many"
)

print("DESPUÉS DE ORDER_ITEMS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE ORDER_ITEMS
Filas: 113425
Pedidos únicos: 99441


In [36]:
df_master = df_master.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE PRODUCTS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE PRODUCTS
Filas: 113425
Pedidos únicos: 99441


In [38]:
df_master = df_master.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE SELLERS")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE SELLERS
Filas: 113425
Pedidos únicos: 99441


In [39]:
df_master = df_master.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

print("DESPUÉS DE CATEGORY TRANSLATION")
print("Filas:", len(df_master))
print("Pedidos únicos:", df_master["order_id"].nunique())

DESPUÉS DE CATEGORY TRANSLATION
Filas: 113425
Pedidos únicos: 99441


### Construcción y alcance del DataFrame maestro

Para la construcción del DataFrame maestro se realizaron uniones (`merge`) progresivas tomando como base la tabla `olist_orders_dataset`, donde cada registro representa un pedido. Primero se incorporó la información de los clientes mediante `customer_id` y posteriormente la información de los productos y vendedores a través de `order_items`, utilizando `product_id` y `seller_id`, respectivamente. Finalmente, se incorporó la traducción de las categorías mediante `product_category_name`. Se utilizaron uniones de tipo `left` para conservar los 99.441 pedidos originales, incluso cuando estos no contaban con información asociada en alguna de las tablas. Como resultado, el DataFrame pasó de 99.441 filas a 113.425 debido a que un mismo pedido puede contener varios productos. Las tablas de reseñas (`reviews`) y pagos (`payments`) no se incorporaron directamente al DataFrame maestro, ya que pueden contener múltiples registros asociados a un mismo pedido y podrían alterar su granularidad; estas serán agregadas posteriormente a nivel de `order_id` según las necesidades de cada análisis. La tabla de geolocalización tampoco se incorporó al DataFrame maestro debido a su gran cantidad de registros y a que un mismo código postal puede aparecer múltiples veces. Durante el proyecto, este DataFrame se utilizará como base para los análisis relacionados con pedidos, productos, categorías y vendedores, teniendo siempre en cuenta su granularidad de pedido-producto. Cuando un análisis requiera contar pedidos, clientes u otras entidades, se utilizarán identificadores únicos mediante `nunique()`, y cuando sea necesario obtener variables a nivel de pedido se realizarán las agregaciones correspondientes antes del análisis.

# limpieza de los datos

In [40]:
missing = df_master.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columnas con valores faltantes:")
print(missing)

Columnas con valores faltantes:
order_delivered_customer_date    3229
product_category_name_english    2402
product_description_lenght       2378
product_photos_qty               2378
product_name_lenght              2378
product_category_name            2378
order_delivered_carrier_date     1968
product_weight_g                  793
product_length_cm                 793
product_width_cm                  793
product_height_cm                 793
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
seller_city                       775
seller_zip_code_prefix            775
seller_state                      775
order_approved_at                 161
dtype: int64


In [41]:

missing_delivery = df_master[
    df_master["order_delivered_customer_date"].isna()
]

print("Pedidos con fecha de entrega faltante:")
print(missing_delivery["order_id"].nunique())

print("\nEstados de estos pedidos:")
print(
    missing_delivery
    .groupby("order_status")["order_id"]
    .nunique()
    .sort_values(ascending=False)
)

Pedidos con fecha de entrega faltante:
2965

Estados de estos pedidos:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: order_id, dtype: int64
